In [1]:
import pandas as pd
import numpy as np
import os
os.makedirs('../data', exist_ok=True)

df = pd.read_csv('../data/creditcard.csv')

print(f"Original shape: {df.shape}")

# ── Remove outliers in Amount (keep fraud cases intact) ──────
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR    # lenient: 3x IQR to keep real fraud
df_clean = df[df['Amount'] <= upper_bound].copy()
print(f"After outlier removal: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")

# ── Feature Engineering ─────────────────────────────────────
# Log-transform Amount (reduces right skew)
df_clean['Amount_Log'] = np.log1p(df_clean['Amount'])

# Time features
df_clean['Hour']       = (df_clean['Time'] // 3600) % 24
df_clean['Is_Night']   = ((df_clean['Hour'] >= 22) | (df_clean['Hour'] <= 6)).astype(int)
df_clean['Is_Morning'] = ((df_clean['Hour'] >= 6) & (df_clean['Hour'] < 12)).astype(int)

# Transaction size flags
df_clean['Small_Txn']  = (df_clean['Amount'] < 50).astype(int)
df_clean['Large_Txn']  = (df_clean['Amount'] > 500).astype(int)
df_clean['Zero_Amount']= (df_clean['Amount'] == 0).astype(int)

print()
print("NEW FEATURES CREATED:")
new_feats = ['Amount_Log','Hour','Is_Night','Is_Morning','Small_Txn','Large_Txn','Zero_Amount']
for f in new_feats:
    print(f"  {f}: mean={df_clean[f].mean():.4f}")

Original shape: (20000, 31)
After outlier removal: (19786, 31)
Rows removed: 214

NEW FEATURES CREATED:
  Amount_Log: mean=3.8522
  Hour: mean=11.4161
  Is_Night: mean=0.3770
  Is_Morning: mean=0.2490
  Small_Txn: mean=0.4649
  Large_Txn: mean=0.0000
  Zero_Amount: mean=0.0001


In [2]:
# Final feature set — exclude target and intermediate columns
FEATURES = [col for col in df_clean.columns
            if col not in ['Class', 'Time', 'Amount_Bracket', 'Hour']]

X = df_clean[FEATURES].copy()
y = df_clean['Class'].copy()

print(f"Total features:    {len(FEATURES)}")
print(f"Total samples:     {len(X):,}")
print(f"Fraud cases:       {y.sum():,} ({y.mean()*100:.3f}%)")
print(f"Legitimate cases:  {(y==0).sum():,}")
print(f"Missing values:    {X.isnull().sum().sum()}")

# Save clean data
df_clean[FEATURES + ['Class']].to_csv('../data/fraud_clean.csv', index=False)
print()
print("Saved: data/fraud_clean.csv")
print(f"Features: {FEATURES}")

Total features:    35
Total samples:     19,786
Fraud cases:       32 (0.162%)
Legitimate cases:  19,754
Missing values:    0

Saved: data/fraud_clean.csv
Features: ['Amount', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount_Log', 'Is_Night', 'Is_Morning', 'Small_Txn', 'Large_Txn', 'Zero_Amount']
